## ロジスティック回帰で簡単なモデルを作成

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator,
)

import mlflow
import mlflow.spark


# ============================================================
# 1. 設定
# ============================================================

SOURCE_TABLE = "workspace.bank.silver_transaction_monthly"

EXPERIMENT_NAME = "/Shared/bank_next_month_overdue_prediction"

# Unity Catalog上のモデル名
REGISTERED_MODEL_NAME = (
    "workspace.bank.next_month_overdue_logistic_regression"
)

# MLflow 3では通常デフォルトですが、明示しておく
mlflow.set_registry_uri("databricks-uc")

mlflow.set_experiment(EXPERIMENT_NAME)


# ============================================================
# 2. データ読み込み
# ============================================================

df = spark.table(SOURCE_TABLE)

required_columns = [
    "customer_id",
    "transaction_month",
    "monthly_inflow",
    "monthly_outflow",
    "deposit_balance",
    "loan_balance",
    "overdue_days",
]

missing_columns = [
    column_name
    for column_name in required_columns
    if column_name not in df.columns
]

if missing_columns:
    raise ValueError(
        f"必要な列がありません: {missing_columns}"
    )


# ============================================================
# 3. 翌月延滞ラベル作成
# ============================================================

customer_month_window = (
    Window
    .partitionBy("customer_id")
    .orderBy("transaction_month")
)

df_labeled = (
    df
    .withColumn(
        "next_month_overdue_days",
        F.lead("overdue_days").over(customer_month_window)
    )
    .withColumn(
        "label",
        F.when(
            F.col("next_month_overdue_days") > 0,
            F.lit(1.0)
        ).otherwise(F.lit(0.0))
    )
)

# 最終月は翌月の実績がないため、学習対象から除外
df_train = (
    df_labeled
    .filter(F.col("next_month_overdue_days").isNotNull())
)


# ============================================================
# 4. 特徴量
# ============================================================

feature_cols = [
    "monthly_inflow",
    "monthly_outflow",
    "deposit_balance",
    "loan_balance",
    "overdue_days",
]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="raw_features",
    handleInvalid="skip",
)

# 金額系のスケールが大きく異なるため標準化
scaler = StandardScaler(
    inputCol="raw_features",
    outputCol="features",
    withMean=True,
    withStd=True,
)

lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    probabilityCol="probability",
    rawPredictionCol="rawPrediction",
    predictionCol="prediction",
    maxIter=50,
    regParam=0.01,
    elasticNetParam=0.0,
)

pipeline = Pipeline(
    stages=[
        assembler,
        scaler,
        lr,
    ]
)


# ============================================================
# 5. ラベル件数確認
# ============================================================

label_counts = {
    float(row["label"]): int(row["count"])
    for row in df_train.groupBy("label").count().collect()
}

print("ラベル件数:", label_counts)

if 0.0 not in label_counts or 1.0 not in label_counts:
    raise ValueError(
        "label=0とlabel=1の両方が必要です。"
        f"現在の件数: {label_counts}"
    )


# ============================================================
# 6. MLflowで学習・記録
# ============================================================

with mlflow.start_run(
    run_name="logistic_regression_next_month_overdue"
) as run:

    # -------------------------------
    # パラメータ記録
    # -------------------------------

    mlflow.log_param("source_table", SOURCE_TABLE)
    mlflow.log_param("target", "next_month_overdue")
    mlflow.log_param("feature_columns", ",".join(feature_cols))
    mlflow.log_param("training_rows", df_train.count())

    mlflow.log_param("max_iter", 50)
    mlflow.log_param("reg_param", 0.01)
    mlflow.log_param("elastic_net_param", 0.0)

    mlflow.log_param(
        "negative_label_count",
        label_counts.get(0.0, 0)
    )
    mlflow.log_param(
        "positive_label_count",
        label_counts.get(1.0, 0)
    )

    # -------------------------------
    # 学習
    # -------------------------------

    model = pipeline.fit(df_train)

    # 学習データに対する評価
    predictions_train = model.transform(df_train)

    # -------------------------------
    # 評価指標
    # -------------------------------

    auc_evaluator = BinaryClassificationEvaluator(
        labelCol="label",
        rawPredictionCol="rawPrediction",
        metricName="areaUnderROC",
    )

    pr_auc_evaluator = BinaryClassificationEvaluator(
        labelCol="label",
        rawPredictionCol="rawPrediction",
        metricName="areaUnderPR",
    )

    accuracy_evaluator = MulticlassClassificationEvaluator(
        labelCol="label",
        predictionCol="prediction",
        metricName="accuracy",
    )

    precision_evaluator = MulticlassClassificationEvaluator(
        labelCol="label",
        predictionCol="prediction",
        metricName="weightedPrecision",
    )

    recall_evaluator = MulticlassClassificationEvaluator(
        labelCol="label",
        predictionCol="prediction",
        metricName="weightedRecall",
    )

    f1_evaluator = MulticlassClassificationEvaluator(
        labelCol="label",
        predictionCol="prediction",
        metricName="f1",
    )

    metrics = {
        "training_roc_auc":
            auc_evaluator.evaluate(predictions_train),

        "training_pr_auc":
            pr_auc_evaluator.evaluate(predictions_train),

        "training_accuracy":
            accuracy_evaluator.evaluate(predictions_train),

        "training_weighted_precision":
            precision_evaluator.evaluate(predictions_train),

        "training_weighted_recall":
            recall_evaluator.evaluate(predictions_train),

        "training_f1":
            f1_evaluator.evaluate(predictions_train),
    }

    mlflow.log_metrics(metrics)

    # -------------------------------
    # モデルを記録し、Unity Catalogへ登録
    # -------------------------------

    from mlflow.models import infer_signature

    # 入力例
    input_example = (
        df_train
        .select(*feature_cols)
        .limit(5)
        .toPandas()
    )

    # Sparkモデルで実際に推論
    signature_output_df = (
        model.transform(
            spark.createDataFrame(input_example)
        )
        .select(
            "prediction",
            "probability",
        )
    )

    # MLflowのシグネチャ推論用にPandas化
    signature_output = signature_output_df.toPandas()

    # 入出力スキーマを作成
    signature = infer_signature(
        input_example,
        signature_output,
    )

    model_info = mlflow.spark.log_model(
        spark_model=model,
        artifact_path="model",
        registered_model_name=REGISTERED_MODEL_NAME,
        input_example=input_example,
        signature=signature,
        dfs_tmpdir="/Volumes/workspace/bank/vol/mlflow_tmp",
    )

    run_id = run.info.run_id

    print(f"MLflow Run ID: {run_id}")
    print(f"登録モデル名: {REGISTERED_MODEL_NAME}")
    print(f"モデルURI: {model_info.model_uri}")
    print("評価指標:", metrics)